# 高级爬虫技巧

本 notebook 演示一些高级的爬虫技术和最佳实践。

## 学习目标
- 并发爬取
- 错误处理和重试
- 代理使用
- 反爬虫应对

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
import time
import random

## 1. 错误处理和重试机制

In [ ]:
def fetch_with_retry(url, max_retries=3, timeout=10):
    """带重试机制的请求函数"""
    for attempt in range(max_retries):
        try:
            response = requests.get(url, timeout=timeout)
            response.raise_for_status()
            return response
        except requests.exceptions.RequestException as e:
            print(f"尝试 {attempt + 1}/{max_retries} 失败: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)  # 指数退避
            else:
                print(f"最终失败: {url}")
                return None

# 测试
response = fetch_with_retry("https://quotes.toscrape.com/")
if response:
    print(f"成功获取，状态码: {response.status_code}")

## 2. 并发爬取

In [ ]:
def scrape_page(page_num):
    """爬取单个页面"""
    url = f"https://quotes.toscrape.com/page/{page_num}/"
    response = fetch_with_retry(url)
    
    if not response:
        return []
    
    soup = BeautifulSoup(response.text, 'lxml')
    quotes = []
    
    for quote in soup.find_all('div', class_='quote'):
        quotes.append({
            '名言': quote.find('span', class_='text').text,
            '作者': quote.find('small', class_='author').text,
            '页码': page_num
        })
    
    return quotes

# 并发爬取多个页面
pages = range(1, 6)  # 爬取前 5 页
all_quotes = []

with ThreadPoolExecutor(max_workers=3) as executor:
    futures = {executor.submit(scrape_page, page): page for page in pages}
    
    for future in tqdm(as_completed(futures), total=len(futures), desc="爬取进度"):
        quotes = future.result()
        all_quotes.extend(quotes)

print(f"共爬取 {len(all_quotes)} 条数据")

## 3. 随机延迟和请求头轮换

In [ ]:
USER_AGENTS = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36',
    'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36'
]

def get_random_headers():
    """获取随机请求头"""
    return {
        'User-Agent': random.choice(USER_AGENTS),
        'Accept': 'text/html,application/xhtml+xml',
        'Accept-Language': 'zh-CN,zh;q=0.9',
    }

def random_delay(min_sec=1, max_sec=3):
    """随机延迟"""
    time.sleep(random.uniform(min_sec, max_sec))

# 示例使用
for i in range(3):
    headers = get_random_headers()
    print(f"请求 {i+1}: {headers['User-Agent'][:50]}...")
    random_delay(0.5, 1.5)

## 4. 会话管理

In [ ]:
class SmartScraper:
    """智能爬虫类"""
    
    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update(get_random_headers())
    
    def get(self, url, **kwargs):
        """带重试的 GET 请求"""
        return fetch_with_retry(url)
    
    def close(self):
        """关闭会话"""
        self.session.close()

# 使用示例
scraper = SmartScraper()
response = scraper.get("https://quotes.toscrape.com/")
print(f"状态码: {response.status_code if response else 'Failed'}")
scraper.close()

## 5. 数据验证和清洗

In [ ]:
# 转换为 DataFrame
df = pd.DataFrame(all_quotes)

# 数据验证
print("数据概览:")
print(f"总行数: {len(df)}")
print(f"缺失值: {df.isnull().sum().sum()}")
print(f"重复行: {df.duplicated().sum()}")

# 去重
df_clean = df.drop_duplicates()
print(f"\n去重后: {len(df_clean)} 行")

df_clean.head()

## 6. 保存结果

In [ ]:
# 保存为多种格式
df_clean.to_csv('../data/quotes_advanced.csv', index=False, encoding='utf-8-sig')
df_clean.to_json('../data/quotes_advanced.json', orient='records', force_ascii=False, indent=2)
df_clean.to_excel('../data/quotes_advanced.xlsx', index=False)

print("数据已保存为 CSV, JSON 和 Excel 格式")